In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mitigación de errores con redes tensoriales (TEM): Una Función (Qiskit Function) por Algorithmiq
> **Note:** Las Qiskit Functions (Funciones de Qiskit) son una característica experimental que está disponible solamente para los usuarios del Plan Premium, Plan Flex y Plan On-Prem (vía IBM Quantum Platform API) de IBM Quantum&reg;. Se encuentran en estado de vista previa sujeta a cambios.


<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página se desarrolló utilizando los siguientes requisitos.
Recomendamos usar estas versiones o más recientes.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Descripción general
El método de mitigación de errores con redes tensoriales (TEM) de Algorithmiq es un algoritmo híbrido cuántico-clásico diseñado para realizar la mitigación de ruido enteramente en la etapa de posprocesamiento clásico. Con TEM, el usuario puede calcular los valores esperados de observables mitigando los inevitables errores inducidos por el ruido que ocurren en el hardware cuántico, con mayor precisión y eficiencia de costos, lo que lo convierte en una opción altamente atractiva para investigadores cuánticos y profesionales de la industria.

El método consiste en construir una red tensorial que representa la inversa del canal de ruido global que afecta el estado del procesador cuántico, y luego aplicar el mapa a los resultados de mediciones informacionalmente completas adquiridos del estado ruidoso para obtener estimadores no sesgados de los observables.

Como ventaja, TEM aprovecha las mediciones informacionalmente completas para dar acceso a un amplio conjunto de valores esperados mitigados de observables y tiene un costo de muestreo óptimo en el hardware cuántico, como se describe en Filippov et al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), y Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). El costo de medición se refiere al número de mediciones adicionales requeridas para realizar una mitigación de errores eficiente, un factor crítico en la viabilidad de las computaciones cuánticas. Por lo tanto, TEM tiene el potencial de habilitar la ventaja cuántica en escenarios complejos, como aplicaciones en los campos del caos cuántico, la física de muchos cuerpos, la dinámica de Hubbard y las simulaciones de química de moléculas pequeñas.

Las principales características y beneficios de TEM se pueden resumir como:

1.  **Costo de muestreo óptimo**: TEM es óptimo con respecto a los [límites teóricos](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601), lo que significa que ningún método puede lograr un menor costo de muestreo. En otras palabras, TEM requiere el número mínimo de mediciones adicionales para realizar la mitigación de errores. Esto a su vez significa que TEM usa un tiempo de ejecución cuántico mínimo.
2.  **Rentabilidad**: Dado que TEM maneja la mitigación de ruido enteramente en la etapa de posprocesamiento, no hay necesidad de agregar circuitos extra al computador cuántico, lo que no solo hace la computación más económica sino que también reduce el riesgo de introducir errores adicionales debido a las imperfecciones de los dispositivos cuánticos.
3.  **Estimación de múltiples observables**: Gracias a las mediciones informacionalmente completas, TEM estima eficientemente múltiples observables con los mismos datos de medición del computador cuántico.
4.  **Mitigación de errores de lectura**: La función TEM de Qiskit también incluye un [método propietario de mitigación de errores de lectura](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154) capaz de reducir significativamente los errores de lectura después de una breve ejecución de calibración.
5.  **Precisión**: TEM mejora significativamente la precisión y fiabilidad de las simulaciones cuánticas digitales, haciendo los algoritmos cuánticos más precisos y confiables.
## Descripción
La función TEM permite obtener valores esperados mitigados para múltiples observables en un circuito cuántico con un costo de muestreo mínimo. El circuito se mide con una medida de operador con valor positivo informacionalmente completa (IC-POVM), y los resultados de medición recolectados se procesan en un computador clásico. Esta medición se usa para realizar los métodos de redes tensoriales y construir un mapa de inversión de ruido. La función aplica un mapa que invierte completamente todo el circuito ruidoso usando redes tensoriales para representar las capas ruidosas.

![Esquema de TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Estimación mitigada de errores de un observable O mediante el posprocesamiento de resultados de medición del procesador cuántico ruidoso. U y N denotan una operación cuántica ideal y el mapa de ruido asociado, que puede ser generalmente no local (y extendido a las cajas grises). D representa un tensor de operadores que son duales a los efectos en la medición IC. El módulo de mitigación de ruido M es una red tensorial que se contrae eficientemente desde el medio hacia afuera. La primera iteración de la contracción se representa con la línea punteada púrpura, la segunda con la línea discontinua, y la tercera con la línea sólida.")

Una vez que los circuitos se envían a la función, se transpilan y optimizan para minimizar el número de capas con puertas de dos qubits (las puertas más ruidosas en los dispositivos cuánticos). El ruido que afecta las capas se aprende a través de [Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner) usando un modelo de ruido sparse Pauli-Lindblad como se describe en E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

El modelo de ruido es una descripción precisa del ruido en el dispositivo, capaz de capturar características sutiles, incluyendo la interferencia entre qubits (cross-talk). Sin embargo, el ruido en los dispositivos puede fluctuar y derivar, y el ruido aprendido podría no ser preciso en el momento en que se realiza la estimación. Esto podría resultar en resultados imprecisos.
## Empezar
Autentícate usando tu [clave API de IBM Quantum Platform](http://quantum.cloud.ibm.com/), y selecciona la función TEM de la siguiente manera. (Este fragmento asume que ya has [guardado tu cuenta](/guides/functions#install-qiskit-functions-catalog-client) en tu entorno local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Ejemplo
El siguiente fragmento muestra un ejemplo donde se usa TEM para calcular los valores esperados de un observable dado un circuito cuántico simple.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Usa las APIs de Qiskit Serverless para verificar el estado de tu carga de trabajo de Qiskit Function:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Puedes obtener los resultados de la siguiente manera:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** El valor esperado para el circuito sin ruido para el operador dado debería ser aproximadamente `0.18409094298943401`.
## Entradas
**Parámetros**

Nombre | Tipo | Descripción | Requerido | Valor por defecto | Ejemplo
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | Un iterable de objetos tipo PUB (bloque unificado de primitivas), tales como tuplas `(circuit, observables)` o `(circuit, observables, parameters, precision)`. Consulta [Descripción general de PUBs](/guides/primitive-input-output#overview-of-pubs) para más información. Si se pasa un circuito que no es ISA, se transpilará con configuraciones óptimas. Si se pasa un circuito ISA, no se transpilará; en este caso, el observable debe estar definido sobre toda la QPU. | Sí | N/A | (circuit, observables)
`backend_name` | str | Nombre del backend para realizar la consulta.| No | Si no se proporciona, se usará el backend menos ocupado. | "ibm_fez"
`options` | dict | Opciones de entrada. Consulta la sección `Opciones` para más detalles. | No | Consulta la sección `Opciones` para más detalles.| {"max_bond_dimension": 100\}

> **Caution:** TEM actualmente tiene las siguientes limitaciones:
> 
>   - Los circuitos parametrizados no están soportados. El argumento de parámetros debe establecerse en `None` si se especifica la precisión. Esta restricción se eliminará en futuras versiones.
>   - Solo se soportan circuitos sin bucles. Esta restricción se eliminará en futuras versiones.
>   - Las puertas no unitarias, como reset, medición y todas las formas de flujo de control no están soportadas. El soporte para reset se agregará en próximas versiones.
### Opciones
Un diccionario que contiene las opciones avanzadas para TEM. El diccionario puede contener las claves de la siguiente tabla. Si alguna de las opciones no se proporciona, se usará el valor por defecto indicado en la tabla. Los valores por defecto son adecuados para el uso típico de TEM.

Nombre | Opciones | Descripción  | Valor defecto
-- | -- | -- | --
`tem_max_bond_dimension` | int | La dimensión máxima de enlace a usar para las redes tensoriales. | 500 |
`tem_compression_cutoff` | float | El valor de corte a usar para las redes tensoriales. | 1e-16
`compute_shadows_bias_from_observable` | bool | Un indicador booleano que señala si el sesgo para el protocolo de medición de sombras clásicas debe adaptarse al observable del PUB o no. Si es False, se usará el protocolo de sombras clásicas (probabilidad igual de medir Z, X, Y).| False
`shadows_bias` | np.ndarray | El sesgo a usar para el protocolo de medición de sombras clásicas aleatorizado, un arreglo 1d o 2d de tamaño 3 o forma (num_qubits, 3) respectivamente. El orden es ZXY. | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int o `None` | El tiempo máximo de ejecución en la QPU en segundos. Si el tiempo de ejecución excede este valor, el trabajo se cancelará. Si es `None`, se aplicará un límite predeterminado establecido por Qiskit Runtime. | `None`
`num_randomizations` | int | El número de aleatorizaciones a usar para el aprendizaje de ruido y el twirling de puertas. | 32
`max_layers_to_learn` | int | El número máximo de capas únicas a aprender. | 4
`mitigate_readout_error` | bool | Un indicador booleano que señala si se debe realizar la mitigación de errores de lectura o no. | True
`num_readout_calibration_shots` | int | El número de disparos (shots) a usar para la mitigación de errores de lectura. | 10000
`default_precision` | float | La precisión predeterminada a usar para los PUBs cuya precisión no se especifique. |0.02
`seed` | int o `None` | Establece la semilla del generador de números aleatorios para reproducibilidad. Si es `None`, no se establece la semilla. | None
## Salidas
Un [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) de Qiskit que contiene el resultado mitigado por TEM. El resultado para cada PUB se devuelve como un [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) que contiene los siguientes campos:

Nombre |Tipo | Descripción
-- | -- | --
data | DataBin | Un [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) de Qiskit que contiene el observable mitigado por TEM y su error estándar. El DataBin tiene los siguientes campos: <ul><li>`evs`: El valor del observable mitigado por TEM.</li><li>`stds`: El error estándar del observable mitigado por TEM.</li></ul>
metadata | dict | Un diccionario que contiene resultados adicionales. El diccionario contiene las siguientes claves: <ul><li>`"evs_non_mitigated"`: El valor del observable sin mitigación de errores.</li><li>`"stds_non_mitigated"`: El error estándar del resultado sin mitigación de errores.</li><li>`"evs_mitigated_no_readout_mitigation"`: El valor del observable con mitigación de errores pero sin mitigación de errores de lectura.</li><li>`"stds_mitigated_no_readout_mitigation"`: El error estándar del resultado con mitigación de errores pero sin mitigación de errores de lectura.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: El valor del observable sin mitigación de errores pero con mitigación de errores de lectura.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: El error estándar del resultado sin mitigación de errores pero con mitigación de errores de lectura.</li></ul>
## Recuperar mensajes de error
Si el estado de tu carga de trabajo es ERROR, usa job.result() para obtener el mensaje de error de la siguiente manera: